# **Simple RAG System**

## **Downloading Data**

In [82]:
# Central Configuration Dictionary to manage all system parameters
config = {
    "data_dir": "../data",                           # Directory to store raw and cleaned data
    "vector_store_dir": "./vector_store",           # Directory to persist our vector store
    "llm_provider": "openai",                       # The LLM provider we are using
    "reasoning_llm": "gpt-4o",                      # The powerful model for planning and synthesis
    "fast_llm": "gpt-4o-mini",                      # A faster, cheaper model for simpler tasks like the baseline RAG
    "embedding_model": "text-embedding-3-small",    # The model for creating document embeddings
    "reranker_model": "cross-encoder/ms-marco-MiniLM-L-6-v2", # The model for precision reranking
    "max_reasoning_iterations": 7,                  # A safeguard to prevent the agent from getting into an infinite loop
    "top_k_retrieval": 10,                          # Number of documents for initial broad recall
    "top_n_rerank": 3,                              # Number of documents to keep after precision reranking
}

In [83]:
# importing all necessary libraries
from dotenv import load_dotenv
import os
import json
import uuid
import re
from pprint import pprint
from typing import List, Literal, Dict, Optional, TypedDict

# loading all environment variables
load_dotenv()

True

In [84]:
import requests
from bs4 import BeautifulSoup
    
# Custom function downloading the 10-K filing and parsing the raw HTML, and converting it into a clean and structured text format suitable for ingestion by RAG pipeline

def download_and_parse_10k(url, data_path_raw, data_path_clean):
    # check if cleaned url already exits to avoid redownloading of dataset
    if os.path.exists(data_path_clean):
        print(f'Cleaned data already exitst at {data_path_clean}')
        return 
    
    # downloading 10k-filling
    print(f"Downloading 10k-filing from url {url}")
    headers = {
    "User-Agent": "MyRAGResearchBot/1.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    }
    respose = requests.get(url=url, headers=headers)
    respose.raise_for_status()

    with open(data_path_raw, 'w', encoding='utf-8') as f:
        f.write(respose.text)
    print(f"Raw data saved to {data_path_raw}")

    soup = BeautifulSoup(respose.content, 'html.parser')

    final_text = []
    for i in soup.find_all(['p', 'div', 'span']):
        text = i.get_text(strip=True) + '\n\n'
        # normalized = re.sub(r"\s+", " ", text)

        if final_text and text == final_text[-1]:
            continue

        final_text.append(text) 

    text = ' '.join(final_text)

    clean_text = re.sub(r'\n{3,}', '\n\n', text).strip()
    clean_text = re.sub(r'[ \t]{2,}', ' ', clean_text).strip()

    with open(data_path_clean, 'w', encoding='utf-8') as f:
        f.write(clean_text)
    print(f'Cleaned text content extracted and saved to {data_path_clean}')

In [85]:
# Execute the download and parsing function

# URL for NVIDIA's 2023 10-K filing (filed Feb 2023 for fiscal year ending Jan 2023)
url_10k = "https://www.sec.gov/Archives/edgar/data/1045810/000104581023000017/nvda-20230129.htm"
os.makedirs(config["data_dir"], exist_ok=True)
data_path_raw = os.path.join(config["data_dir"], "nvda_10k_2023_raw.html")
data_path_clean = os.path.join(config["data_dir"], "nvda_10k_2023_clean.txt")

print("Downloading and parsing NVIDIA's 2023 10-K filing...")
download_and_parse_10k(url_10k, data_path_raw, data_path_clean)

with open(data_path_clean, 'r', encoding='utf-8') as f:
    print("--- Sample content from cleaned 10-K ---")
    print(f.read(1000) + "...")

Cleaned data already exitst at ../data\nvda_10k_2023_clean.txt
--- Sample content from cleaned 10-K ---
00010458102023FYfalseP3YP4YP5YP1YP3YP1Yhttp://fasb.org/us-gaap/2022#AccruedLiabilitiesCurrenthttp://fasb.org/us-gaap/2022#AccruedLiabilitiesCurrenthttp://fasb.org/us-gaap/2022#AccruedLiabilitiesCurrent00010458102022-01-312023-01-2900010458102022-07-29iso4217:USD00010458102023-02-17xbrli:shares00010458102021-02-012022-01-3000010458102020-01-272021-01-31iso4217:USDxbrli:shares00010458102023-01-2900010458102022-01-300001045810us-gaap:CommonStockMember2020-01-260001045810us-gaap:AdditionalPaidInCapitalMember2020-01-260001045810us-gaap:TreasuryStockMember2020-01-260001045810us-gaap:AccumulatedOtherComprehensiveIncomeMember2020-01-260001045810us-gaap:RetainedEarningsMember2020-01-2600010458102020-01-260001045810us-gaap:RetainedEarningsMember2020-01-272021-01-310001045810us-gaap:AccumulatedOtherComprehensiveIncomeMember2020-01-272021-01-310001045810us-gaap:CommonStockMember2020-01-272021-01

## **Building Simple RAG Application**

In [86]:
# importing required libraries
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# loadin text data
print('Loading and chunking documents...')
loader = TextLoader(data_path_clean, encoding='utf-8')
docs = loader.load()

# splitting text data into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
splitted_docs = text_splitter.split_documents(docs)

print(f"Documents loaded and split into {len(splitted_docs)} chunks.")


Loading and chunking documents...
Documents loaded and split into 571 chunks.


In [87]:
# creating embeddings and vector store

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma

print('Creating baseline vector store...')
# creating embedding model
embeddings = OpenAIEmbeddings(model=config['embedding_model'])

# creating vector store
vector_store = Chroma.from_documents(
    documents=splitted_docs,
    embedding=embeddings
)

# creating retriever
retriever = vector_store.as_retriever(search_kwargs={'k':3})

print(f'Vector store created with {vector_store._collection.count()} embeddings')

Creating baseline vector store...
Vector store created with 1142 embeddings


In [88]:
# creating prompt template

from langchain_core.prompts import ChatPromptTemplate

template = """You are an AI financial analyst. Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

In [89]:
# Creating RAG chain

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# string output parser
str_output_parser = StrOutputParser()

# defining LLM for answering question
llm = ChatOpenAI(model=config['fast_llm'])

# function for joining retrieved text
def join_retrieved_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# basic RAG chain
basic_rag_chain = (
    {'context': retriever | join_retrieved_docs, 'question': RunnablePassthrough()} 
    | prompt
    | llm
    | str_output_parser
)


In [90]:
# checking inference

from rich.console import Console # For pretty-printing output with markdown
from rich.markdown import Markdown

# Initialize the rich console for better output formatting
console = Console()

# Our complex, multi-hop, multi-source query
complex_query_adv = "Based on NVIDIA's 2023 10-K filing, identify their key risks related to competition. Then, find recent news (post-filing, from 2024) about AMD's AI chip strategy and explain how this new strategy directly addresses or exacerbates one of NVIDIA's stated risks."

print("Executing complex query on the baseline RAG chain...")
# Invoke the chain with our challenging query
baseline_result = basic_rag_chain.invoke(complex_query_adv)

console.print("\n--- BASELINE RAG FAILED OUTPUT ---")
# Print the result using markdown formatting for readability
console.print(Markdown(baseline_result))

Executing complex query on the baseline RAG chain...


c:\Users\masan\OneDrive\Desktop\Adaptive Multi-Agent RAG System\.venv\Lib\site-packages\pydantic\v1\main.py:1054: UserWarning: LangSmith now uses UUID v7 for run and trace identifiers. This warning appears when passing custom IDs. Please use: from langsmith import uuid7
            id = uuid7()
Future versions will require UUID v7.
  input_data = validator(cls_, input_data)


--- BASELINE RAG FAILED OUTPUT ---

Based on NVIDIA's 2023 10-K filing, the key risks related to competition include:                                  

 1 Intensity of Competition: NVIDIA faces significant competition from companies providing GPUs, CPUs, DPUs,       
   embedded SoCs, and semiconductor-based high-performance interconnect products. This competition may come from   
   both established giants and newer entrants in the AI computing space.                                           
 2 Resource Disparities: Some competitors have greater marketing, financial, distribution, and manufacturing       
   resources, allowing them to adapt more effectively to customer demands and technological changes.               
 3 Evolving Market Needs: The failure to meet the evolving needs of the industry and markets can adversely impact  
   NVIDIA’s financial results, suggesting a need to stay responsive to market trends and technological             
   advancements.                                                                                                   

To find the recent news on AMD's AI chip strategy from 2024 and its implications on NVIDIA's risks, we can consider
AMD's continued focus on developing advanced chips specifically optimized for AI workloads. For instance, if AMD   
launched new AI-focused GPUs that are competitively priced and provide superior performance or energy efficiency   
compared to NVIDIA's offerings, it would intensify the competitive pressure on NVIDIA, directly addressing the risk
of an increasingly competitive environment.                                                                        

This news could signal to investors that not only is AMD working on innovative products, but it could also threaten
NVIDIA’s market share in AI applications, which is a growing area of demand. This would exacerbate NVIDIA's stated 
risks regarding competition as it highlights the challenges of maintaining leadership in a fast-evolving sector    
with dynamic needs and rapidly advancing technology.